# Tutorial 24: PIDNet Cityscapes

This notebook explains, builds, and runs a Qt5 C++ semantic-segmentation application. It processes camera or video frames with PIDNet on a DEEPX NPU, highlights the large input tensor in the bottom control bar, and provides a compact live slider for the PIDNet argmax scale.

## 1. Processing pipeline

```text
Camera or video frame
        |
        v
Resize to model input and convert BGR to RGB
        |
        v
Asynchronous PIDNet inference -> class logits
        |
        v
Bilinear interpolation at the selected argmax scale
        |
        v
Per-pixel argmax -> Cityscapes color mask -> Qt5 GUI
```

The default model accepts UINT8 `[1, 1024, 2048, 3]` input and returns FLOAT32 `[1, 19, 128, 256]` logits.

## 2. Prerequisites

The system needs a supported DEEPX NPU, the device driver, the DXRT SDK, and a graphical desktop session.

```bash
sudo apt update
sudo apt install -y build-essential cmake pkg-config libopencv-dev qtbase5-dev ffmpeg v4l-utils
```

Check the NPU connection with `dxrt-cli -s`.

In [ ]:
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / "app").is_dir() and (cwd / "README.md").is_file():
    PROJECT_DIR = cwd
else:
    PROJECT_DIR = cwd / "notebooks" / "T24-demo-pidnet-cityscapes"

PROJECT_DIR = PROJECT_DIR.resolve()
APP_DIR = PROJECT_DIR / "app"
ASSETS_DIR = PROJECT_DIR / "assets"

assert APP_DIR.is_dir(), f"Application directory not found: {APP_DIR}"
print(f"Project: {PROJECT_DIR}")
print(f"Application: {APP_DIR}")
print(f"Assets: {ASSETS_DIR}")

## 3. Project layout

C++ code and launch scripts are under `app/`. Downloaded models and videos belong under `assets/`.

In [ ]:
for path in sorted(PROJECT_DIR.rglob("*")):
    if "build" in path.relative_to(PROJECT_DIR).parts:
        continue
    if path.is_file():
        print(path.relative_to(PROJECT_DIR))

## 4. Download and check the resources

This demo references the official [XuJiacong/PIDNet](https://github.com/XuJiacong/PIDNet) project. Its `PIDNet_S_Cityscapes_val.pt` PyTorch checkpoint was converted to DXNN format for DEEPX NPU inference. The application uses the resulting `pidnet_s_cityscapes_val_fixed.dxnn` model.

`get_resources.sh` downloads the resource archive, extracts the model and sample video into `assets/`, and removes the downloaded archive after successful extraction:

```text
assets/
├── models/
│   └── pidnet_s_cityscapes_val_fixed.dxnn
└── videos/
    └── pidnet.mp4
```

In [ ]:
required_resources = [
    ASSETS_DIR / "models" / "pidnet_s_cityscapes_val_fixed.dxnn",
    ASSETS_DIR / "videos" / "pidnet.mp4",
]

for path in required_resources:
    status = "ready" if path.is_file() else "missing"
    print(f"{status:7} {path.relative_to(PROJECT_DIR)}")

missing_resources = [path for path in required_resources if not path.is_file()]

Run the next cell only when resources are missing. Existing files with the same names may be replaced during extraction.

In [ ]:
import subprocess

if missing_resources:
    subprocess.run(
        [str(PROJECT_DIR / "get_resources.sh")],
        cwd=PROJECT_DIR,
        check=True,
    )
else:
    print("All required resources are already available.")

for path in required_resources:
    status = "ready" if path.is_file() else "missing"
    print(f"{status:7} {path.relative_to(PROJECT_DIR)}")

### Optional model inspection

If `dxparse` and the model are available, the next cell prints the input and output tensors. Otherwise, it skips the check without failing.

In [ ]:
import shutil
import subprocess

dxparse = shutil.which("dxparse")
model_path = required_resources[0]
if dxparse is None:
    print("dxparse is not installed; skipping model inspection.")
elif not model_path.is_file():
    print(f"Skipping missing model: {model_path.name}")
else:
    subprocess.run([dxparse, "-m", str(model_path), "-v"], check=True)

## 5. Understanding `pidnet_argmax_scale`

PIDNet produces lower-resolution class logits. Before selecting the most likely class at each pixel, the application bilinearly interpolates those logits. The scale controls the intermediate interpolation size:

- `0.1`: lowest CPU cost and coarser boundaries.
- `0.4`: default balance between speed and detail.
- `1.0`: full-frame argmax resolution and highest CPU cost.

The GUI slider uses values from 0.10 through 1.00 in steps of 0.05. It writes to an atomic value, and each asynchronous completion callback reads that value once before processing a frame. This gives every frame one consistent scale.

In [ ]:
def scaled_argmax_size(logits_size, target_size, scale):
    return max(1, min(target_size, max(logits_size, round(target_size * scale))))

logits_width = 256
frame_width = 1920
for scale in (0.1, 0.4, 1.0):
    width = scaled_argmax_size(logits_width, frame_width, scale)
    print(f"scale={scale:.1f} -> intermediate argmax width={width}")

## 6. C++ code guide

The implementation is in `app/pidnet_cityscapes.cpp`. The next cell locates its main sections.

In [ ]:
source_path = APP_DIR / "pidnet_cityscapes.cpp"
source_lines = source_path.read_text(encoding="utf-8").splitlines()
symbols = [
    "struct Options",
    "preprocess_frame",
    "compute_argmax_mask",
    "render_segmentation",
    "class VideoView",
    "class MainWindow",
    "class FrameMailbox",
    "class InflightLimiter",
    "QSlider(Qt::Horizontal",
    "run_inference",
]

for symbol in symbols:
    line_number = next((i for i, line in enumerate(source_lines, 1) if symbol in line), None)
    print(f"{line_number:4}: {symbol}")

### Main implementation stages

1. `Options` and `parse_args` select a camera or video, the model, camera settings, the initial scale, and overlay opacity.
2. `preprocess_frame` converts BGR to RGB and directly resizes the frame to the model input tensor.
3. `compute_argmax_mask` interpolates every class channel at the current scale and selects the highest-scoring class.
4. `render_segmentation` maps class IDs to Cityscapes colors and blends the mask with the input frame.
5. `MainWindow` places the video above a bottom control bar containing the runtime input shape, compact scale slider, and Exit button.
6. The capture thread submits up to four requests with `RunAsync`, allowing NPU inference and CPU post-processing to overlap.
7. Completion callbacks publish only the newest rendered result to a one-frame mailbox. A Qt timer consumes it without accumulating stale GUI events.
8. OpenCV operations use one internal thread because frame-level callback parallelism already uses the available CPU cores.

## 7. Build

`build.sh` configures a Release build and runs `make` with all available CPU cores. Use `--clean` to remove the old build directory first.

In [ ]:
subprocess.run(["./build.sh"], cwd=APP_DIR, check=True)

In [ ]:
subprocess.run(["./build/pidnet_cityscapes", "--help"], cwd=APP_DIR, check=True)

## 8. Run with a camera

The script requests camera index 0 at 1280 x 720 and 30 FPS. Set `RUN_CAMERA` to `True` only when the NPU, model, camera, and graphical session are ready.

In [ ]:
RUN_CAMERA = False

if RUN_CAMERA:
    subprocess.run([
        "./run_camera.sh", "--camera", "0",
        "--width", "1280", "--height", "720", "--fps", "30",
    ], cwd=APP_DIR, check=True)
else:
    print("Camera execution is disabled. Set RUN_CAMERA = True when the hardware and GUI are ready.")

## 9. Run with a video

`run_video.sh` reads `assets/videos/pidnet.mp4` and loops it. Set `RUN_VIDEO` to `True` after placing the model and video in the expected locations.

In [ ]:
RUN_VIDEO = False

if RUN_VIDEO:
    subprocess.run(["./run_video.sh"], cwd=APP_DIR, check=True)
else:
    print("Video execution is disabled. Set RUN_VIDEO = True when the resources and GUI are ready.")

## 10. Custom commands

Start with another slider value:

```bash
cd app
./run_video.sh --pidnet-argmax-scale 0.7
```

Use another video in windowed mode:

```bash
./build/pidnet_cityscapes --video /path/to/input.mp4 --loop --windowed
```

Select another camera and reduce mask opacity:

```bash
./run_camera.sh --camera 2 --width 1920 --height 1080 --fps 30 --alpha 0.45
```

Use `Esc`, `Q`, or the Exit button to quit. Press `F` to toggle full-screen mode. The default `--inflight 4` is intended for a four-core Raspberry Pi 5; test `--inflight 6` if additional throughput is needed.

## 11. Troubleshooting

- If the model is missing, verify its filename under `assets/models/`.
- If the video is missing, place `pidnet.mp4` under `assets/videos/` or pass another path.
- If the camera cannot be opened, use `v4l2-ctl --list-devices` and try another index.
- If no window appears, verify access to a graphical display.
- If CMake cannot find DXRT, set `DXRT_INSTALLED_DIR` and run `./build.sh --clean`.
- If CPU post-processing is slow, move the Argmax scale slider toward 0.1.